# ARC3 offline training data CPU gate

Uses hashed private training assets, actual native multimodal processor/loss masks and meta-device LoRA placement. CPU only, no model weights loaded or gradients. No score claim. Upstream attribution and licenses retained.

In [ ]:
ARCHIVE_SHA256='10e5df7fbe82f82f4cbf95e923e465b900184e36fa9877fbafdf752e6b549961'
EXPECTED_INPUT_MANIFEST={'files': {'16k/train.jsonl': {'sha256': 'fd73b3802e886bd4dc6c443ac24388f3c491f7720b6213ded89e4bc5889d71c5', 'bytes': 7031316}, '16k/validation.jsonl': {'sha256': 'c8527dde162787d496e842a6a4684cf692e9490211b8ce520176ade02c8ea452', 'bytes': 2815735}, '32k/train.jsonl': {'sha256': 'd3e8c271f7fc67c3c5ff72019946ff5a46bf4f90f0bbc973b08da58c44c19efe', 'bytes': 18380809}, '32k/validation.jsonl': {'sha256': 'c40c876828c0338036fdde844e1d47f7d66f575dfc22e5e9721a33dc39489d74', 'bytes': 4696845}, 'wheels/accelerate-1.15.0-py3-none-any.whl': {'sha256': '97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec', 'bytes': 394295}, 'wheels/fla_core-0.6.0-py3-none-any.whl': {'sha256': 'b0708b70e0b119cd58a0a5146ea56cb175cef6ed969d8ac60267e3e077ce3ebd', 'bytes': 1085221}, 'wheels/flash_linear_attention-0.6.0-py3-none-any.whl': {'sha256': '6dec9735f8454523c13553622e98aff30152dde6586b4ac7d88010fbb7a06b5f', 'bytes': 403549}, 'wheels/peft-0.21.2-py3-none-any.whl': {'sha256': '106ab6077ff72c54d21577f9af5970e34bac7582cb14209f7b2b511e322a4eae', 'bytes': 835427}, 'arc3_training_collator.py': {'sha256': 'e27f3881e0b83ab3f3cdc4da216158ed5dd69e6807891e5ea925702766170858', 'bytes': 2460}, 'split.json': {'sha256': 'ce36006c553278e8282b9e8ca05bfa0b6f6b39a9ffc831d03dd9419e760e5525', 'bytes': 1233}, 'base-requirements.lock': {'sha256': '4b0ee12848c1e7c3a51b0819de3a1ae63a967313fd1f62b9e2ca6b9c82b8da02', 'bytes': 4187}}, 'purpose': 'Private offline ARC3 training/evaluation inputs. No pretrained weights, credentials, hidden games or test-game examples.', 'attribution': 'Agent/prompt source Daniel Franzen, Jeroen Cottaar and Tufa Labs, Apache2.0. Our own recorded Qwen agent outputs. Upstream dependency licenses remain inside wheels. Preserve applicable original notices; no blanket relicensing of third-party material.'}
BASE_LOCK_SHA256='4b0ee12848c1e7c3a51b0819de3a1ae63a967313fd1f62b9e2ca6b9c82b8da02'
MODEL_PATH='/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1'
BF16_PATH='/kaggle/input/models/dangkhoa2016/qwen-qwen3-8-27b/transformers/default/1'
EXPECTED_TARGET_METADATA={'config.json': {'sha256': '74227dd615bf1ea975aa676bdf355a0379858c12f394b5365cd9dfa5fc2c70bc', 'size': 51350}, 'tokenizer.json': {'sha256': '0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3', 'size': 12809320}, 'tokenizer_config.json': {'sha256': 'b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27', 'size': 17928}, 'chat_template.jinja': {'sha256': 'c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041', 'size': 8952}, 'preprocessor_config.json': {'sha256': '27225450ac9c6529872ee1924fcb0962ff5634834f817040f444118116f4e516', 'size': 390}}
from pathlib import Path
from hashlib import sha256
import json,os,subprocess,sys,zipfile,urllib.request
work=Path('/kaggle/working');bundle=work/'arc3-training-dependencies';bundle.mkdir(exist_ok=True)
roots=[Path('/kaggle/input/datasets/dfranzen/pennyroyal-v253'),Path('/kaggle/input/pennyroyal-v253')]
root=next(p for p in roots if p.is_dir());wheels=root/'wheels';lock=root/'requirements.lock'
assert sha256(lock.read_bytes()).hexdigest()==BASE_LOCK_SHA256,'Base lock changed'
environment=dict(os.environ,UV_OFFLINE='1',UV_PYTHON_DOWNLOADS='never',PYTHONNOUSERSITE='1',
    HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',USE_TF='0',USE_TORCHVISION='0')
environment.pop('PYTHONPATH',None)
venv=Path('/tmp/arc3-training-cpu-env');uv=Path('/tmp/arc3-training-uv');python=venv/'bin/python'
uv_wheels=list(wheels.glob('uv-*.whl'));assert len(uv_wheels)==1
with zipfile.ZipFile(uv_wheels[0]) as archive:
    names=[n for n in archive.namelist() if Path(n).name=='uv' and not n.endswith('/')];assert len(names)==1
    uv.write_bytes(archive.read(names[0]));uv.chmod(0o755)
def run(arguments):
    result=subprocess.run(list(map(str,arguments)),env=environment,text=True,capture_output=True)
    print(result.stdout[-3500:],flush=True)
    if result.returncode:print(result.stderr[-7000:],flush=True);raise RuntimeError('Isolated CPU command failed')
import tarfile
input_roots=[Path('/kaggle/input/datasets/kunalsinghh25/arc3-training-pilot-inputs-v1'),Path('/kaggle/input/arc3-training-pilot-inputs-v1')]
assets=work/'training-assets';assets.mkdir(exist_ok=True)
archives=[root/'training-pilot-inputs.tar.gz' for root in input_roots if (root/'training-pilot-inputs.tar.gz').is_file()]
if archives:
    assert len(archives)==1
    archive=archives[0]
    assert sha256(archive.read_bytes()).hexdigest()==ARCHIVE_SHA256,'Private input archive changed'
    tar=tarfile.open(archive,'r:gz')
    manifest=json.loads(tar.extractfile('manifest.json').read())
    def read_asset(name):
        member=tar.getmember(name)
        if not member.isfile():raise ValueError('Invalid archive member')
        return tar.extractfile(member).read()
    input_asset_mode='archive-sha256-and-file-sha256'
else:
    # Kaggle automatically expanded this uploaded tar.gz. The API file listing
    # verifies this exact prefix; validate every original manifest/file hash.
    roots=[root/'training-pilot-inputs' for root in input_roots if (root/'training-pilot-inputs/manifest.json').is_file()]
    if len(roots)!=1:raise ValueError('Pinned expanded assets are not uniquely mounted: '+str(roots))
    source_root=roots[0].resolve();manifest=json.loads((source_root/'manifest.json').read_text())
    def read_asset(name):
        source=(source_root/name).resolve()
        if not source.is_relative_to(source_root) or not source.is_file():raise ValueError('Invalid expanded asset path')
        return source.read_bytes()
    input_asset_mode='kaggle-expanded-exact-manifest-and-file-sha256'
assert manifest==EXPECTED_INPUT_MANIFEST
try:
    for name,record in manifest['files'].items():
        destination=(assets/name).resolve()
        if not destination.is_relative_to(assets.resolve()):raise ValueError('Asset escapes output root')
        content=read_asset(name)
        assert len(content)==record['bytes'] and sha256(content).hexdigest()==record['sha256'],name
        destination.parent.mkdir(parents=True,exist_ok=True);destination.write_bytes(content)
finally:
    if archives:tar.close()
(assets/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('TRAINING_ASSET_INTEGRITY',input_asset_mode,flush=True)
model=Path(MODEL_PATH);bf16=Path(BF16_PATH)
assert sha256((bf16/'tokenizer.json').read_bytes()).hexdigest()==EXPECTED_TARGET_METADATA['tokenizer.json']['sha256'],'BF16 training tokenizer differs from deployment target'
bc=json.loads((bf16/'config.json').read_text());tc=json.loads((model/'config.json').read_text())
for key in ['vocab_size','hidden_size','num_hidden_layers','num_attention_heads','num_key_value_heads']:
    assert bc['text_config'][key]==tc['text_config'][key],key
assert bc['architectures']==tc['architectures']==['Qwen3_5ForConditionalGeneration']
for wheel in (assets/'wheels').glob('*.whl'):(bundle/wheel.name).symlink_to(wheel)
run([uv,'venv','--python',sys.executable,venv])
run([uv,'pip','install','--python',python,'--no-index','--find-links',wheels,'-r',lock])
# Public, pinned pure Python wheels only. Existing Torch/CUDA/Transformers are retained.
run([uv,'pip','install','--python',python,'--no-index','--no-deps',*sorted(bundle.glob('*.whl'))])
run([uv,'pip','check','--python',python])
rows=[]
for split in ['train','validation']:
    rows.extend(dict(context=16384,record=json.loads(line)) for line in (assets/'16k'/f'{split}.jsonl').read_text().splitlines())
    candidates=[json.loads(line) for line in (assets/'32k'/f'{split}.jsonl').read_text().splitlines()]
    rows.extend(dict(context=32768,record=row) for row in sorted(candidates,key=lambda r:r['text_tokens']+r['visual_expansion_tokens'],reverse=True)[:8])
(work/'selected-training-rows.json').write_text(json.dumps(rows))
probe=work/'actual_training_data_probe.py'
probe.write_text('MODEL_PATH=\'/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1\'\nEXPECTED_METADATA={\'config.json\': {\'sha256\': \'74227dd615bf1ea975aa676bdf355a0379858c12f394b5365cd9dfa5fc2c70bc\', \'size\': 51350}, \'tokenizer.json\': {\'sha256\': \'0997f410c57a1f4e53b09e4be8f4a172d90edd9564368fb0847030937229b9f3\', \'size\': 12809320}, \'tokenizer_config.json\': {\'sha256\': \'b11349aafa7cdc6a320767cf7ceb29ed82f7eda5d65e8e0819e76f0ce947bf27\', \'size\': 17928}, \'chat_template.jinja\': {\'sha256\': \'c3cf9e34abf4f9e36c2d72165aa9c132d3e2a725b6c2586aaa3a8af9d7a81041\', \'size\': 8952}, \'preprocessor_config.json\': {\'sha256\': \'27225450ac9c6529872ee1924fcb0962ff5634834f817040f444118116f4e516\', \'size\': 390}}\nCOLLATOR_SOURCE=\'"""Single-example native multimodal collation with exact assistant-only loss."""\\nimport base64\\nfrom hashlib import sha256\\nfrom io import BytesIO\\n\\nfrom PIL import Image\\n\\n\\ndef collate_example(record,processor,max_tokens=32768):\\n    """Requires the actual target processor. No teacher code is executed."""\\n    messages=record[\\\'messages\\\']\\n    if messages[-1][\\\'role\\\']!=\\\'assistant\\\':raise ValueError(\\\'Final supervised assistant reply required\\\')\\n    kwargs=record[\\\'template_kwargs\\\']\\n    render=processor.tokenizer.apply_chat_template\\n    prefix=render(messages[:-1],tools=record[\\\'tools\\\'],tokenize=False,add_generation_prompt=True,**kwargs)\\n    text=render(messages,tools=record[\\\'tools\\\'],tokenize=False,add_generation_prompt=False,**kwargs)\\n    if (not text.startswith(prefix) or len(prefix)!=record[\\\'assistant_start_char\\\']\\n            or sha256(text.encode()).hexdigest()!=record[\\\'rendered_text_sha256\\\']):\\n        raise ValueError(\\\'Native target rendering differs from the frozen data audit\\\')\\n    images=[]\\n    for message in messages[:-1]:\\n        content=message.get(\\\'content\\\')\\n        if not isinstance(content,list):continue\\n        for part in content:\\n            if part.get(\\\'type\\\')!=\\\'image_url\\\':continue\\n            url=part[\\\'image_url\\\'][\\\'url\\\']\\n            if not url.startswith(\\\'data:image/png;base64,\\\'):raise ValueError(\\\'Only embedded PNG observations allowed\\\')\\n            image=Image.open(BytesIO(base64.b64decode(url.split(\\\',\\\',1)[1],validate=True))).convert(\\\'RGB\\\')\\n            images.append(image)\\n    batch=processor(text=[text],images=images or None,return_tensors=\\\'pt\\\',padding=False,add_special_tokens=False)\\n    prompt=processor(text=[prefix],images=images or None,return_tensors=\\\'pt\\\',padding=False,add_special_tokens=False)\\n    ids=batch[\\\'input_ids\\\'];prompt_ids=prompt[\\\'input_ids\\\'];length=prompt_ids.shape[-1]\\n    if ids.shape[0]!=1 or ids.shape[-1]>max_tokens:raise ValueError(\\\'Training context limit exceeded\\\')\\n    if not ids[:,:length].equal(prompt_ids):raise ValueError(\\\'Actual image-expanded token prefix differs\\\')\\n    labels=ids.clone();labels[:,:length]=-100\\n    if int((labels!=-100).sum())<16:raise ValueError(\\\'No substantial supervised reply\\\')\\n    batch[\\\'labels\\\']=labels\\n    return batch,dict(actual_input_tokens=ids.shape[-1],masked_prompt_tokens=length,\\n        supervised_tokens=int((labels!=-100).sum()),images=len(images),\\n        boundary=\\\'Actual processor tensor/loss-mask checks only; not a backward pass or model-quality result.\\\')\\n\'\nimport json,torch,hashlib\nfrom pathlib import Path\nfrom transformers import AutoProcessor,AutoConfig,AutoModelForImageTextToText\nfrom accelerate import init_empty_weights\nfrom peft import LoraConfig,get_peft_model\ntorch.set_num_threads(4)\nroot=Path(MODEL_PATH)\nfor name,expected in EXPECTED_METADATA.items():\n    assert hashlib.sha256((root/name).read_bytes()).hexdigest()==expected[\'sha256\'],name\nprocessor=AutoProcessor.from_pretrained(root,local_files_only=True,trust_remote_code=False)\nnamespace={};exec(COLLATOR_SOURCE,namespace)\ncollate=namespace[\'collate_example\']\nrows=json.loads(Path(\'/kaggle/working/selected-training-rows.json\').read_text())\nchecks=[];errors=[]\nfor index,entry in enumerate(rows):\n    record=entry[\'record\']\n    assert record[\'metadata\'][\'split\'] in [\'train\',\'validation\']\n    try:\n        tensors,info=collate(record,processor,max_tokens=entry[\'context\'])\n        assert tensors[\'input_ids\'].device.type==\'cpu\'\n        assert tensors[\'labels\'].device.type==\'cpu\'\n        assert int((tensors[\'labels\']!=-100).sum())==record[\'supervised_text_tokens\']\n        assert info[\'masked_prompt_tokens\']>0\n        checks.append(dict(example=record[\'example_id\'],split=record[\'metadata\'][\'split\'],**info))\n        del tensors\n    except Exception as exc:\n        errors.append(dict(example=record[\'example_id\'],type=type(exc).__name__,error=str(exc)[:400]))\n    if index%25==0:print(\'COLLATION_PROGRESS\',index,len(rows),flush=True)\nconfig=AutoConfig.from_pretrained(root,local_files_only=True,trust_remote_code=False)\nwith init_empty_weights():model=AutoModelForImageTextToText.from_config(config,torch_dtype=torch.bfloat16)\ntargets=[name for name,module in model.named_modules() if isinstance(module,torch.nn.Linear)\n    and \'language_model.layers.\' in name and \'.self_attn.\' in name\n    and name.rsplit(\'.\',1)[-1] in [\'q_proj\',\'k_proj\',\'v_proj\',\'o_proj\']]\nassert len(targets)==64,(\'Expected16 full-attention layers *4 projections\',len(targets))\nmodel=get_peft_model(model,LoraConfig(r=16,lora_alpha=32,target_modules=targets,\n    lora_dropout=0.0,bias=\'none\',task_type=\'CAUSAL_LM\'))\ntrainable=[(name,p.numel()) for name,p in model.named_parameters() if p.requires_grad]\nassert trainable and all(\'lora_\' in name for name,count in trainable)\nassert all(\'language_model.layers.\' in name and \'.self_attn.\' in name for name,count in trainable)\nreport=dict(passed=not errors,examples_expected=len(rows),examples_passed=len(checks),errors=errors,\n    tensor_checks=checks,lora_target_modules=targets,lora_trainable_parameters=sum(n for _,n in trainable),\n    cuda_available=torch.cuda.is_available(),\n    boundary=\'Actual CPU processor tensors, frozen render/prefix agreement, assistant-only labels and meta-model LoRA placement. No full checkpoint, CUDA backward step, training quality, adapter reload or serving compatibility established.\')\nPath(\'/kaggle/working/training-data-probe.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\nprint(json.dumps({k:v for k,v in report.items() if k not in [\'tensor_checks\',\'lora_target_modules\']},indent=2))\nassert report[\'passed\'],\'Actual training data processor gate failed; inspect retained report\'\n')
run([python,probe])
run([python,'-c','from fla.ops.gated_delta_rule import chunk_gated_delta_rule; assert callable(chunk_gated_delta_rule); print("FLA_CPU_IMPORT_PASSED")'])
(bundle/'manifest.json').write_text(json.dumps(dict(files={p.name:sha256(p.read_bytes()).hexdigest() for p in sorted(bundle.glob('*.whl'))},base_lock_sha256=BASE_LOCK_SHA256),indent=2)+'\n')
